# Les 11 - Agent-tot-Agent (A2A) Protocol


## Installatie


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Wat is het A2A-protocol?

Het **Agent-to-Agent (A2A) protocol** is een open standaard die AI-agenten in staat stelt om te communiceren,
elkaar te ontdekken en samen te werken — zelfs wanneer ze op verschillende frameworks zijn gebouwd of worden gehost
door verschillende diensten.

Belangrijke concepten:

- **Ontdekking** – Agenten publiceren een *Agent Card* die hun mogelijkheden beschrijft, waardoor het
  voor andere agenten (of orkestrators) eenvoudig is om de juiste specialist voor een taak te vinden.
- **Berichtuitwisseling** – Agenten wisselen gestructureerde berichten uit via een gemeenschappelijk protocol, zodat een
  verzoek van de ene agent begrepen en uitgevoerd kan worden door een andere, ongeacht de interne
  implementatie.
- **Taaklevenscyclus** – A2A definieert statussen zoals *ingediend*, *bezig*, *voltooid* en
  *mislukt*, waardoor de orkestrator volledig inzicht heeft in hoe een gedelegeerde taak vordert.

In deze les simuleren we A2A-achtige samenwerking door drie gespecialiseerde reisagenten
te koppelen in een workflow waarbij elke agent zijn expertise inbrengt en resultaten doorgeeft aan de volgende.


## Specialiseerde reisagenten creëren


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Samenwerking tussen meerdere agenten via workflow

We verbinden de drie agenten in een opeenvolgende workflow die A2A-berichtuitwisseling weerspiegelt:

1. **CurrencyExchangeAgent** ontvangt het gebruikersverzoek en geeft valuta-advies.
2. **ActivityPlannerAgent** ontvangt de verrijkte context en voegt activiteitenaanbevelingen toe.
3. **TravelManagerAgent** synthetiseert beide inputs tot een definitieve reisbriefing.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Begrijpen van A2A in productie

In een productieomgeving ontgrendelt het A2A-protocol krachtige cross-service scenario's:

| Mogelijkheid | Beschrijving |
|---|---|
| **Cross-framework interoperabiliteit** | Een agent gebouwd met één framework kan taken delegeren aan een agent gebouwd met elk ander A2A-compatibel framework, wat echte interoperabiliteit tussen organisaties mogelijk maakt. |
| **Servicegrenzen** | Agents kunnen in afzonderlijke microservices, cloudregio's of zelfs verschillende organisaties leven terwijl ze toch naadloos samenwerken. |
| **Dynamische ontdekking** | Een orkestrator kan tijdens runtime een Agent Card-register raadplegen om de best geschikte specialist voor een bepaalde subtaken te vinden. |
| **Streaming & pushmeldingen** | A2A ondersteunt Server-Sent Events (SSE) voor realtime voortgangsupdates en pushmeldingen voor langlopende taken. |

De workflow die we hierboven hebben gebouwd is een vereenvoudigde, in-process versie van dit patroon. In een echte
implementatie zou elke agent een HTTP-endpoint exposen, een Agent Card publiceren, en communiceren
via het A2A JSON-RPC-protocol.


## Summary

In this lesson you learned:

1. **What the A2A protocol is** — an open standard for agent-to-agent discovery, messaging,
   and task management.
2. **How to create specialized agents** — a Currency Exchange agent, an Activity Planner agent,
   and a Travel Manager orchestrator.
3. **How to wire agents into a workflow** — using `WorkflowBuilder` to model sequential
   message passing between agents.
4. **How A2A works in production** — enabling cross-framework, cross-service collaboration
   with dynamic discovery and streaming updates.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
